# 00 — Thiết lập Colab + train tiếp `trba_ft` (bản tải từ tài khoản A, ghi chú để chạy ở tài khoản B)

**Đã làm ở A (05/11):** ô 1–5. Ô 5 tạo run `runs/trba_ft/2026-11-05_08-28-47` (giờ UTC, tức 15:28 giờ VN),
train được **8/20 epoch** (khoảng 32 phút/epoch trên T4), đang chạy epoch thứ 9 (577/801 batch) thì
mất phiên vì tài khoản A hết hạn mức GPU. Checkpoint nằm trên Drive của A.

**Cách làm: không chép.** B làm việc thẳng trên thư mục `trademark_data` của A qua một lối tắt giữ
nguyên tên, nên trên Colab B mọi đường dẫn (`/content/drive/MyDrive/trademark_data/...`) trùng đúng
với lúc A train. Checkpoint mới ghi thẳng vào Drive của A; file mới do B tạo thì tính vào dung lượng
của B.

**Làm trước trên drive.google.com, theo đúng thứ tự:**

1. **Tài khoản A** — mở `trademark_data/runs/trba_ft/2026-11-05_08-28-47/checkpoints/`. Tạo thư mục
   `trademark_data/backup_trba_ft_8epoch/` rồi **di chuyển** (không phải sao chép) 3 file `epoch=…ckpt` vào đó.
   **Để nguyên `last.ckpt`.** Lý do:
   - Khi train ở B, Lightning xoá dần checkpoint cũ khỏi top-3. 3 file này thuộc sở hữu của A, mà
     Drive thường chỉ cho chủ sở hữu xoá file: B xoá không được thì train dừng vì lỗi. File không
     còn ở chỗ cũ thì Lightning bỏ qua bước xoá (đã kiểm trong code Lightning).
   - Trạng thái 8 epoch được giữ nguyên ở `backup_trba_ft_8epoch/`, làm bản dự phòng.
2. **Tài khoản A** — chuột phải thư mục `trademark_data` → Chia sẻ → email của B, quyền
   **Người chỉnh sửa** → Gửi.
3. **Tài khoản B** — trong "Drive của tôi" phải **chưa có** thư mục nào tên `trademark_data` (có thì:
   trống → xoá; có dữ liệu → đổi tên). Rồi "Được chia sẻ với tôi" → chuột phải `trademark_data` →
   Sắp xếp → Thêm lối tắt → Drive của tôi. **Giữ nguyên tên `trademark_data`.**
4. Không để A và B cùng train run này một lúc.

**Mở bằng tài khoản B:** colab.research.google.com (đăng nhập B) → Tệp → Tải sổ tay lên → chọn file
này. Runtime → Change runtime type → **T4 GPU**.

**Thứ tự chạy, mỗi phiên, KHÔNG dùng "Run all":** 1 → 2 → 3 → 4 → **B1** (kiểm quyền ghi) → 6 →
**B2** (train tiếp). Ô 5 là lịch sử của A, đã vô hiệu hoá: chạy lại nó là tạo run mới, train lại từ
epoch 1.

Lưu ý điều khoản Colab về việc dùng nhiều tài khoản: xem `docs/nhat_ky_chay.md` mục 3b.

In [ ]:
# [Ô 1] Mount Drive + biến dùng chung. Chạy MỖI phiên.
# - Hộp thoại cấp quyền: chọn TÀI KHOẢN B.
# - DATA_ROOT = /content/drive/MyDrive/trademark_data = LỐI TẮT tới thư mục của A (bước 3 ở đầu sổ
#   tay), nên đường dẫn trùng đúng với lúc A train.
# - Ô gốc có os.makedirs(DATA_ROOT): đã bỏ. Nếu lối tắt chưa hiện (chưa đồng bộ), makedirs sẽ âm thầm
#   tạo một thư mục trademark_data TRỐNG của B, che mất lối tắt. Thay bằng kiểm tra rồi dừng.
# - BRANCH ở ô này không dùng: ô 2 đặt lại BRANCH = "16-fix-review".
# - Repo private thì REPO_URL phải có token: https://<token>@github.com/...
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = "https://github.com/nguyenlamvu123/major_proj_1.git"  # repo private: https://<token>@github.com/...
BRANCH = "main"                                                   # nhánh đã merge các thay đổi cần chạy
os.environ["DATA_ROOT"] = "/content/drive/MyDrive/trademark_data"

# --- thêm cho tài khoản train tiếp ---
TK = "B"        # TÊN TÀI KHOẢN ĐANG CHẠY: "B" ở phiên 1, "C" ở phiên 2, ... (mỗi tài khoản một mốc riêng ở ô B1)
GIO = "2026-11-05_08-28-47"                                         # run do ô 5 tạo ở A
RUN = f"/content/drive/MyDrive/trademark_data/runs/trba_ft/{GIO}"   # qua lối tắt, cùng đường dẫn như ở A
if not os.path.exists(f"{RUN}/checkpoints/last.ckpt"):
    raise SystemExit(
        f"Không thấy {RUN}/checkpoints/last.ckpt.\n"
        "- Lối tắt trademark_data chưa có, sai tên, hoặc chưa đồng bộ: chờ vài phút rồi chạy\n"
        "  drive.mount('/content/drive', force_remount=True) và chạy lại ô này.\n"
        "- Hoặc Drive của B có sẵn một thư mục trademark_data riêng che mất lối tắt: xoá/đổi tên nó.")
print("OK: thấy run của A qua lối tắt:", RUN)

Mounted at /content/drive


In [ ]:
# [Ô 2] Clone code nhánh 16-fix-review. Chạy MỖI phiên (máy ảo mới không còn /content).
# - Ra khỏi thư mục trước khi xoá, rồi clone sạch (ô gốc của notebook bỏ qua clone nếu thư mục đã có).
# - Dòng `git log` phải in một commit từ 58c7227 trở đi (ở A hôm 05/11 là 4a3b70a, cũ hơn).
#   Code mới đọc được hydra.run.dir/data.root_dir trong 05_train.sh; checkpoint cũ vẫn dùng được.
%cd /content
BRANCH = "16-fix-review"
!rm -rf /content/major_proj_1
!git clone -b {BRANCH} {REPO_URL} /content/major_proj_1
%cd /content/major_proj_1
!git log --oneline -1

/content
Cloning into '/content/major_proj_1'...
remote: Enumerating objects: 639, done.
remote: Counting objects: 100% (639/639), done.
remote: Compressing objects: 100% (291/291), done.
remote: Total 639 (delta 307), reused 624 (delta 292), pack-reused 0 (from 0)
Receiving objects: 100% (639/639), 714.28 KiB | 12.99 MiB/s, done.
Resolving deltas: 100% (307/307), done.
/content/major_proj_1
4a3b70a (HEAD -> 16-fix-review, origin/16-fix-review) 16-fix-review: flag undertrained models, archive stale reports, clamp char_accuracy after first full run


In [ ]:
# [Ô 3] Cài thư viện + clone parseq vào third_party/. Chạy MỖI phiên.
# - Dòng đỏ "pip's dependency resolver ... google-colab requires ipykernel ..." là bình thường
#   (đã gặp ở A), không ảnh hưởng train.
# - Cuối phải in: [00_get_parseq] third_party/parseq @ 1902db0
# torch/torchvision có sẵn trên Colab; requirements chỉ cài phần còn thiếu
!pip install -q -r requirements.txt && pip install -q -e .
!bash scripts/00_get_parseq.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.4.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter_client<8.0,>=5.2.0, but you have jupyter-client 8.10.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires notebook<7.0,>=5.7.6, but you have notebook 7.6.4 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for trademark-ocr (pyproject.toml) ... done
Cloning into '/content/major_proj_1/third_party/parseq'...
remote: Enumerating objec

In [ ]:
# [Ô 4] Kiểm GPU + vài test nhanh. Chạy MỖI phiên.
# - Phải thấy "GPU 0: Tesla T4". Báo lỗi nvidia-smi = runtime chưa chọn GPU (Runtime → Change runtime
#   type → T4 GPU) hoặc tài khoản không được cấp GPU lúc này. Không có GPU thì ĐỪNG chạy B2.
# - Test phải "45 passed".
!nvidia-smi -L
!python -m pytest -q -x tests/test_charset.py tests/test_parseq_contracts.py tests/test_lmdb.py

GPU 0: Tesla T4 (UUID: GPU-b75adf31-302d-bfde-be71-b3848d3e5a69)
.............................................                            [100%]
45 passed in 25.90s


In [ ]:
# [Ô 5] LỊCH SỬ — ô train ĐÃ CHẠY Ở A ngày 05/11. ĐÃ VÔ HIỆU HOÁ, KHÔNG chạy lại.
# - RUN dùng datetime.now(), nên chạy lại là tạo run MỚI và train lại từ epoch 1 (phí 8 epoch).
#   Train tiếp thì dùng ô B2.
# - Output bên dưới là của A: run 2026-11-05_08-28-47, 8 epoch xong (dòng "Epoch 7/20 801/801" — Lightning đếm
#   từ 0 và hiện /20 vì SWA cộng thêm một lượt), mất phiên ở "Epoch 8/20 577/801".
# - Dòng "[05_train] checkpoint -> /content/lmdb_root/runs/..." trong output là IN SAI của code cũ
#   (4a3b70a không đọc hydra.run.dir khi in). Checkpoint thật được ghi vào RUN trên Drive.
# - DATA_ROOT=/content/lmdb_root ở dòng train là để train.py đọc LMDB từ đĩa máy ảo
#   (data.root_dir mặc định = $DATA_ROOT/lmdb); ô B2 giữ y như vậy.
#
# Mã gốc (đã thêm # ở đầu mỗi dòng):
# MODEL = "trba_ft"
# # !mkdir -p /content/lmdb_root && tar -xf $DATA_ROOT/lmdb_trainval.tar -C /content/lmdb_root
# # !bash scripts/05_train.sh trba_ft --max-epochs 20 --num-workers 2 data.root_dir=/content/lmdb_root/lmdb

# !mkdir -p /content/lmdb_root && tar -xf /content/drive/MyDrive/trademark_data/lmdb_trainval.tar -C /content/lmdb_root
# !ls /content/lmdb_root/lmdb/train/trademark /content/lmdb_root/lmdb/val    # phải thấy data.mdb lock.mdb

# import datetime
# RUN = "/content/drive/MyDrive/trademark_data/runs/trba_ft/" + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# print(RUN)
# !DATA_ROOT=/content/lmdb_root bash scripts/05_train.sh trba_ft --max-epochs 20 --num-workers 2 hydra.run.dir={RUN}

/content/lmdb_root/lmdb/train/trademark:
data.mdb  lock.mdb

/content/lmdb_root/lmdb/val:
data.mdb  lock.mdb
/content/drive/MyDrive/trademark_data/runs/trba_ft/2026-11-05_08-28-47
[05_train] model=trba_ft config=tm_trba_ft overrides: trainer.max_epochs=20 data.num_workers=2 hydra.run.dir=/content/drive/MyDrive/trademark_data/runs/trba_ft/2026-11-05_08-28-47
[05_train] PYTHON=/usr/bin/python3  DATA_ROOT=/content/lmdb_root
[05_train] checkpoint -> /content/lmdb_root/runs/trba_ft/<timestamp>/
/content/major_proj_1/third_party/parseq/train.py:64: DeprecationWarning: torch.get_autocast_gpu_dtype() is deprecated. Please use torch.get_autocast_dtype('cuda') instead. (Triggered internally at /pytorch/torch/csrc/autograd/init.cpp:881.)
  config.trainer.precision = 'bf16-mixed' if torch.get_autocast_gpu_dtype() is torch.bfloat16 else '16-mixed'
/usr/local/lib/python3.13/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm

In [ ]:
# [Ô B1] Kiểm thư mục run và quyền ghi TRƯỚC khi train. Chạy MỖI phiên, chỉ mất vài giây.
# Dừng ở dòng nào thì sửa theo thông báo, đừng chạy B2.
# (a) PHIÊN ĐẦU: checkpoints/ không còn file epoch=… nào (bước 1 ở đầu sổ tay đã di chuyển cả 3 file
#     của A sang backup_trba_ft_8epoch/; phiên 2 trở đi: tài khoản phiên trước chuyển file của NÓ sang thư mục
#     backup khác). Qua kiểm thì ghi file mốc <TK>_da_kiem_checkpoint.txt; các phiên sau CÙNG TK thấy
#     mốc thì bỏ qua (a), vì lúc đó file epoch=… trong checkpoints/ là do B tạo — kể cả epoch=7-…:
#     khi train tiếp, Lightning chấm val lại bước vừa khôi phục và ghi lại file của epoch đó.
# (b) B tạo được và xoá được file của chính mình trong thư mục run (checkpoint mới, log mới).
# (c) B ghi đè được một file nhỏ của A (hparams.yaml) theo ĐÚNG cách Lightning ghi đè last.ckpt
#     (ghi ra file tạm rồi chuyển đè). Ghi lại đúng nội dung cũ, nên file không đổi.
#     Cần kiểm vì nếu bước này hỏng, Lightning 2.6 có thể nuốt lỗi PermissionError: train vẫn chạy mà
#     last.ckpt không được cập nhật.
# Lỗi ở (b) hoặc (c): quyền chia sẻ chưa phải Người chỉnh sửa, hoặc Drive không cho — chuyển sang cách
# chép run sang Drive của B (docs/nhat_ky_chay.md mục 3b, cách 2).
import os, re, fsspec
ck = f"{RUN}/checkpoints"
files = sorted(os.listdir(ck))
print("checkpoints/:", files)
moc = f"{RUN}/{TK}_da_kiem_checkpoint.txt"
ep = [f for f in files if f.startswith("epoch=")]
if os.path.exists(moc):
    print(f"(a) bỏ qua: không phải phiên đầu của tài khoản {TK} (có mốc", open(moc).read().strip() + f"); file epoch=… hiện có do {TK} tạo")
elif ep:
    raise SystemExit(f"(a) Phiên đầu của tài khoản {TK} mà checkpoints/ còn file của tài khoản trước: {ep}\n    → CHỦ của các file đó (A ở phiên 1, B ở phiên 2, ...) DI CHUYỂN chúng sang một thư mục backup trong trademark_data/, rồi chạy lại ô này.\n    (Đặt sai TK ở ô 1, vd vẫn để B trong phiên của C, thì (a) không bắt được: kiểm lại TK.)")
else:
    import datetime
    with open(moc, "w") as f:
        f.write("B kiểm xong lúc " + datetime.datetime.now().strftime("%Y-%m-%d %H:%M UTC"))
    print("(a) OK (phiên đầu): checkpoints/ chỉ còn last.ckpt; đã ghi mốc", moc.rsplit("/", 1)[-1])

t = f"{RUN}/thu_ghi_B.txt"
with open(t, "w") as f:
    f.write("thu")
os.remove(t)
print(f"(b) OK: tạo + xoá file của {TK} trong thư mục run")

p = next(x for x in (f"{RUN}/hparams.yaml", f"{RUN}/config/config.yaml") if os.path.exists(x))
data = open(p, "rb").read()
fs, up = fsspec.core.url_to_fs(p)
with fs.transaction:              # y như lightning_fabric.utilities.cloud_io._atomic_save
    with fs.open(up, "wb") as f:
        f.write(data)
assert open(p, "rb").read() == data, "nội dung sau khi ghi đè bị khác"
print("(c) OK: ghi đè được file của A theo cách Lightning lưu checkpoint:", p.rsplit("/", 1)[-1])

In [ ]:
# [Ô 6] Kiểm last.ckpt trước khi train tiếp. Chạy MỖI phiên, trước B2.
# - "đã xong N epoch": số epoch mà last.ckpt chứa. Lần đầu ở B mong đợi 8 | 6408 (801 bước/epoch).
#   N có thể nhỏ hơn epoch đang chạy lúc mất phiên: Lightning 2.6 chỉ ghi lại last.ckpt ở epoch lọt
#   top-3 val_accuracy. Train tiếp từ đó vẫn đúng, chỉ chạy lại các epoch sau nó.
# - "dirpath trong ckpt" phải là RUN + "/./checkpoints" (lối tắt giữ tên nên đường dẫn trùng với A).
# - "top-3": lần đầu ở B là 3 file epoch≤7 đã chuyển sang backup_trba_ft_8epoch/ — đúng, Lightning thấy chúng
#   không còn thì không xoá.
# - torch.load báo lỗi (last.ckpt ghi dở lúc mất phiên): chép file epoch=…ckpt có số epoch lớn nhất
#   (phiên đầu: trong backup_trba_ft_8epoch/) VÀO {RUN}/checkpoints/, rồi ở B2 đặt CKPT = đường dẫn bản chép đó.
#   Bản chép phải nằm trong {RUN}/checkpoints/: train.py lấy thư mục run từ ckpt_path (lên 2 cấp).
!ls -la {RUN}/checkpoints
import torch
c = torch.load(f"{RUN}/checkpoints/last.ckpt", map_location="cpu", weights_only=False)
print("đã xong", c["epoch"] + 1, "epoch | global_step", c["global_step"])
mc = [v for k, v in c.get("callbacks", {}).items() if "ModelCheckpoint" in k]
if mc:
    print("dirpath trong ckpt:", mc[0].get("dirpath"))
    print("top-3 val_accuracy:", {p.rsplit("/", 1)[-1]: round(float(s), 2) for p, s in mc[0].get("best_k_models", {}).items()})

In [ ]:
# [Ô B2] TRAIN TIẾP run cũ. Chạy MỖI phiên, sau ô 1–4, B1 và 6.
# - Giống lệnh ở ô 5 của A, chỉ khác: hydra.run.dir = RUN CŨ (không phải giờ hiện tại), thêm ckpt_path.
#   ckpt_path khôi phục trọng số, optimizer, lịch learning rate, bộ đếm epoch và chạy nốt tới 20 epoch.
# - KHÔNG đổi --max-epochs 20 và KHÔNG thêm --batch-size (mặc định 64): đổi là lịch OneCycle lệch với
#   bản đã lưu → lỗi "Tried to step ... times" hoặc learning rate sai. --num-workers đổi được.
# - Chạy đúng khi output có đủ:
#     [05_train] checkpoint -> /content/drive/MyDrive/trademark_data/runs/trba_ft/2026-11-05_08-28-47/
#     Restored all states from the checkpoint at ...
#     thanh tiến trình ở "Epoch 7/20" (Lightning chấm val lại epoch vừa khôi phục, vài phút) rồi
#     "Epoch 8/20" (epoch thứ 9; hiện /20 vì SWA cộng một lượt). Thấy "Epoch 0/20" là SAI: dừng ngay.
# - Ô này chạy thì các ô khác phải xếp hàng chờ. Muốn biết checkpoint có được ghi không: sau ~35 phút
#   (hết epoch đầu ở B) mở drive.google.com → trademark_data/runs/trba_ft/<GIO>/checkpoints/: thường sẽ
#   có file epoch=8-… mới và last.ckpt có giờ sửa mới (chỉ khi epoch đó lọt top-3 val_accuracy).
# - 12 epoch còn lại × ~32 phút ≈ 6,5 giờ. Mất phiên giữa chừng: chạy lại ô 1 → 2 → 3 → 4 → B1 → 6 → B2.
# - Xong 20 epoch: mang run về máy local (ô cuối sổ tay).
CKPT = f"{RUN}/checkpoints/last.ckpt"   # last.ckpt hỏng: xem ghi chú ô 6
!mkdir -p /content/lmdb_root && tar -xf /content/drive/MyDrive/trademark_data/lmdb_trainval.tar -C /content/lmdb_root
!ls /content/lmdb_root/lmdb/train/trademark /content/lmdb_root/lmdb/val    # phải thấy data.mdb lock.mdb
!DATA_ROOT=/content/lmdb_root bash scripts/05_train.sh trba_ft --max-epochs 20 --num-workers 2 hydra.run.dir={RUN} ckpt_path={CKPT}

## Xong 20 epoch: mang về máy local

Chi tiết: `docs/nhat_ky_chay.md` mục 3, "Mang về máy local". Run nằm trong thư mục `trademark_data` của
A, nên tải từ A hay B đều được.

1. Tải `trademark_data/runs/trba_ft/2026-11-05_08-28-47/`: `checkpoints/last.ckpt`, `config/`, `hparams.yaml`,
   **mọi** file `events.out.tfevents.*` (mỗi lần train tiếp sinh thêm một file) và **tên** các file
   `checkpoints/epoch=…ckpt`.
2. Ở máy local đặt vào `data/runs/trba_ft/2026-11-05_08-28-47/`. Nếu val_accuracy tốt nhất ≥ 1% thì
   `touch data/runs/trba_ft/2026-11-05_08-28-47/DONE`.
3. Lúc máy rảnh: `bash scripts/09_full_pipeline_cpu.sh --only eval` (chấm test + vẽ đồ thị).
4. `trademark_data/backup_trba_ft_8epoch/` (3 checkpoint của 8 epoch đầu) xoá được sau khi đã eval xong.